# SOMOSPIE Python/GDAL Demo: Alabama, June 2019

This notebook runs the current Python/GDAL preprocessing workflow without R and without rasterio. It prepares model-ready training and evaluation CSVs for Alabama using ESA CCI Soil Moisture for June 2019 and terrain rasters from `/media/volume/aitechx_vol1/terrain_parameters/AL`.

The shell equivalent is `run_pipeline.sh` in this same directory.

In [ ]:
from pathlib import Path
import csv
import math
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "SOMOSPIE",
    Path("/home/exouser/git/chandlerdev/SOMOSPIE_revisions/SOMOSPIE"),
]
SOMOSPIE_DIR = next(path for path in candidate_dirs if (path / "code" / "preprocessing").exists())
PREPROC_DIR = SOMOSPIE_DIR / "code" / "preprocessing"
FETCH_DIR = SOMOSPIE_DIR / "data_readers" / "satellite"
sys.path.insert(0, str(PREPROC_DIR))

from add_topos import add_topos
from coarsify import coarsify
from create_shape import create_shape
from crop_to_shape import crop_to_shape
from extract_SM_monthly import extract_sm_monthly
from make_raster_stack import make_raster_stack
from reproject_raster import reproject_raster

print(f"SOMOSPIE_DIR={SOMOSPIE_DIR}")

In [ ]:
STATE_NAMES = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "DC": "District of Columbia",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois",
    "IN": "Indiana", "IA": "Iowa", "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana",
    "ME": "Maine", "MD": "Maryland", "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota",
    "MS": "Mississippi", "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
}

STATE = os.environ.get("STATE", "AL").upper()
YEAR = int(os.environ.get("YEAR", "2019"))
MONTH = int(os.environ.get("MONTH", "6"))
MONTH_PAD = f"{MONTH:02d}"

DATA_ROOT = Path(os.environ.get("SOMOSPIE_DATA_ROOT", "/media/volume/aitechx_vol1"))
TERRAIN_ROOT = Path(os.environ.get("TERRAIN_ROOT", str(DATA_ROOT / "terrain_parameters")))
TERRAIN_DIR = Path(os.environ.get("TERRAIN_DIR", str(TERRAIN_ROOT / STATE)))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", str(DATA_ROOT / "SOMOSPIE_demo")))
RUN_DIR = OUTPUT_ROOT / f"{STATE}_{YEAR}_{MONTH_PAD}"

ESA_DIR = RUN_DIR / "ESA_CCI"
SHAPE_DIR = RUN_DIR / "shapes"
COARSE_DIR = RUN_DIR / "terrain_coarse"
REPROJECTED_DIR = RUN_DIR / "terrain_wgs84"
STACK_DIR = RUN_DIR / "stacks"
TRAIN_DIR = RUN_DIR / "training"
EVAL_DIR = RUN_DIR / "evaluation"
PRED_DIR = RUN_DIR / "predictions"
STATE_BOUNDARY_DIR = RUN_DIR / "state_boundaries"

STATE_BOUNDARY_URL = os.environ.get(
    "STATE_BOUNDARY_URL",
    "https://www2.census.gov/geo/tiger/GENZ2025/shp/cb_2025_us_state_500k.zip",
)
COARSIFY_FACTOR = int(os.environ.get("COARSIFY_FACTOR", "10"))
FORCE = os.environ.get("FORCE", "0") == "1"
RUN_MODEL = os.environ.get("RUN_MODEL", "0") == "1"
MODEL = os.environ.get("MODEL", "rf")

for directory in [ESA_DIR, SHAPE_DIR, COARSE_DIR, REPROJECTED_DIR, STACK_DIR, TRAIN_DIR, EVAL_DIR, PRED_DIR, STATE_BOUNDARY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

STATE_NAME = STATE_NAMES.get(STATE)
if STATE_NAME is None:
    raise ValueError(f"STATE must be a two-letter state code known to this notebook; got {STATE!r}")
if not TERRAIN_DIR.exists():
    raise FileNotFoundError(f"Terrain directory not found: {TERRAIN_DIR}")

print(f"State: {STATE_NAME} ({STATE})")
print(f"Terrain input: {TERRAIN_DIR}")
print(f"Run directory: {RUN_DIR}")

In [ ]:
def download_state_boundaries():
    """Download the current Census state boundary shapefile if needed."""
    env_vector = os.environ.get("SOMOSPIE_STATE_VECTOR")
    if env_vector and Path(env_vector).exists():
        return Path(env_vector)

    existing = sorted(STATE_BOUNDARY_DIR.rglob("cb_*_us_state_500k.shp"))
    if existing:
        return existing[-1]

    zip_path = STATE_BOUNDARY_DIR / Path(STATE_BOUNDARY_URL).name
    if FORCE or not zip_path.exists():
        print(f"Downloading state boundaries: {STATE_BOUNDARY_URL}")
        urllib.request.urlretrieve(STATE_BOUNDARY_URL, zip_path)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(STATE_BOUNDARY_DIR)

    matches = sorted(STATE_BOUNDARY_DIR.rglob("cb_*_us_state_500k.shp"))
    if not matches:
        raise FileNotFoundError(f"No state shapefile found after extracting {zip_path}")
    return matches[-1]

state_vector = download_state_boundaries()
os.environ["SOMOSPIE_STATE_VECTOR"] = str(state_vector)
shape_path = SHAPE_DIR / f"{STATE}.geojson"
if FORCE or not shape_path.exists():
    create_shape("STATE", STATE_NAME, shape_path)
print(f"State vector: {state_vector}")
print(f"Region shape: {shape_path}")

In [ ]:
fetch_script = FETCH_DIR / "fetch_soil_moisture.sh"
subprocess.run([str(fetch_script), str(YEAR), str(ESA_DIR)], check=True)

sm_month_tif = ESA_DIR / f"{YEAR}_{MONTH_PAD}_ESA_monthly.tif"
if FORCE or not sm_month_tif.exists():
    extract_sm_monthly(YEAR, ESA_DIR, MONTH, MONTH, sm_month_tif)
print(f"Monthly soil moisture raster: {sm_month_tif}")

In [ ]:
terrain_sources = sorted(
    path for path in TERRAIN_DIR.glob("*.tif")
    if path.is_file() and not path.name.startswith(".")
)
if not terrain_sources:
    raise FileNotFoundError(f"No .tif terrain rasters found in {TERRAIN_DIR}")

reprojected_files = []
band_names = []
for src in terrain_sources:
    name = src.stem
    band_names.append(name)
    if COARSIFY_FACTOR > 1:
        coarse_path = COARSE_DIR / f"{name}_coarse.tif"
        if FORCE or not coarse_path.exists():
            print(f"Coarsifying {src.name} -> {coarse_path.name}")
            coarsify(src, coarse_path, COARSIFY_FACTOR)
        reproject_source = coarse_path
    else:
        reproject_source = src

    reproj_path = REPROJECTED_DIR / f"{name}.tif"
    if FORCE or not reproj_path.exists():
        print(f"Reprojecting {reproject_source.name} -> {reproj_path.name}")
        reproject_raster(reproject_source, reproj_path, "EPSG:4326")
    reprojected_files.append(reproj_path)

stack_tif = STACK_DIR / f"{STATE}_terrain_stack.tif"
if FORCE or not stack_tif.exists():
    make_raster_stack(reprojected_files, stack_tif, strict=True, band_names=band_names)
print(f"Stacked {len(reprojected_files)} terrain raster(s): {stack_tif}")

In [ ]:
sm_points_csv = TRAIN_DIR / f"{STATE}_{YEAR}_{MONTH_PAD}_soil_moisture_points.csv"
train_raw_csv = TRAIN_DIR / f"{STATE}_{YEAR}_{MONTH_PAD}_train_with_covariates_raw.csv"
eval_raw_csv = EVAL_DIR / f"{STATE}_{YEAR}_{MONTH_PAD}_eval_raw.csv"

if FORCE or not sm_points_csv.exists():
    crop_to_shape(sm_month_tif, shape_path, sm_points_csv, 0, [str(MONTH)])
if FORCE or not train_raw_csv.exists():
    add_topos(sm_points_csv, stack_tif, train_raw_csv, band_names)
if FORCE or not eval_raw_csv.exists():
    crop_to_shape(stack_tif, shape_path, eval_raw_csv, 0, band_names)

print(f"Training raw CSV: {train_raw_csv}")
print(f"Evaluation raw CSV: {eval_raw_csv}")

In [ ]:
def is_valid_number(value):
    """Return True when a CSV field is a finite numeric value."""
    if value is None:
        return False
    text = str(value).strip()
    if text == "" or text.upper() in {"NA", "NAN", "NULL", "NONE"}:
        return False
    try:
        return math.isfinite(float(text))
    except ValueError:
        return False


def read_csv_dicts(path):
    """Read a CSV into field names and row dictionaries."""
    with open(path, newline="") as handle:
        reader = csv.DictReader(handle)
        return reader.fieldnames or [], list(reader)


def write_csv_dicts(path, fieldnames, rows):
    """Write selected dictionary rows to CSV."""
    with open(path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def clean_model_tables(train_raw, eval_raw, train_out, eval_out, month):
    """Create model-ready train/eval CSVs with matching finite feature columns."""
    train_fields, train_rows = read_csv_dicts(train_raw)
    eval_fields, eval_rows = read_csv_dicts(eval_raw)
    target_candidates = [str(month), f"{int(month):02d}", f"X{int(month)}", "band_1"]
    target_col = next((name for name in target_candidates if name in train_fields), None)
    if target_col is None:
        raise ValueError(f"Could not find month target column in {train_raw}; fields={train_fields}")

    topo_cols = [name for name in train_fields if name not in {"x", "y", target_col} and name in eval_fields]
    train_keep = []
    for row in train_rows:
        required = ["x", "y", target_col] + topo_cols
        if all(is_valid_number(row.get(name)) for name in required):
            cleaned = {"x": row["x"], "y": row["y"], "z": row[target_col]}
            cleaned.update({name: row[name] for name in topo_cols})
            train_keep.append(cleaned)

    eval_keep = []
    for row in eval_rows:
        required = ["x", "y"] + topo_cols
        if all(is_valid_number(row.get(name)) for name in required):
            eval_keep.append({name: row[name] for name in required})

    if not train_keep:
        raise ValueError("No valid training rows after dropping NA/non-finite values.")
    if not eval_keep:
        raise ValueError("No valid evaluation rows after dropping NA/non-finite values.")

    train_fields_out = ["x", "y", "z"] + topo_cols
    eval_fields_out = ["x", "y"] + topo_cols
    write_csv_dicts(train_out, train_fields_out, train_keep)
    write_csv_dicts(eval_out, eval_fields_out, eval_keep)
    return len(train_keep), len(eval_keep), topo_cols

train_csv = TRAIN_DIR / f"{STATE}_{YEAR}_{MONTH_PAD}_train.csv"
eval_csv = EVAL_DIR / f"{STATE}_{YEAR}_{MONTH_PAD}_eval.csv"
train_count, eval_count, feature_names = clean_model_tables(train_raw_csv, eval_raw_csv, train_csv, eval_csv, MONTH)
print(f"Model-ready training rows: {train_count:,}")
print(f"Model-ready evaluation rows: {eval_count:,}")
print(f"Feature count including x/y: {len(feature_names) + 2}")
print(f"Training CSV: {train_csv}")
print(f"Evaluation CSV: {eval_csv}")

In [ ]:
if RUN_MODEL:
    model_script = SOMOSPIE_DIR / "code" / "modeling" / f"{MODEL}.py"
    pred_csv = PRED_DIR / f"{STATE}_{YEAR}_{MONTH_PAD}_{MODEL}_predictions.csv"
    if MODEL == "rf":
        model_args = ["-maxtree", os.environ.get("RF_MAXTREE", "300"), "-seed", os.environ.get("MODEL_SEED", "42")]
    elif MODEL == "knn":
        model_args = ["-k", os.environ.get("KNN_MAXK", "20"), "-seed", os.environ.get("MODEL_SEED", "42")]
    else:
        raise ValueError(f"Unsupported MODEL={MODEL!r}; expected rf or knn")
    subprocess.run([
        sys.executable, str(model_script),
        "-t", str(train_csv),
        "-e", str(eval_csv),
        "-o", str(pred_csv),
        *model_args,
    ], check=True)
    print(f"Prediction CSV: {pred_csv}")
else:
    print("Model step skipped. Set RUN_MODEL=1 to run rf.py or knn.py after preprocessing.")